# Week 1 — The Agentic Loop, by hand

*Course project: a personal assistant for **Alex, 44**: senior backend developer, full-time job, two kids,
retraining as an **AI engineer** in an evening program while job-hunting. A lot like you, maybe.*

**Goal for today:** build a working agent from scratch with nothing but a model API and a `while` loop.

> An agent is a model in a loop. On each turn the model either **answers** or **asks for a tool**.
> We run the tool, append the result to the conversation, and call the model again.

By the end of this notebook you will have:
1. Seen why a single LLM call is not enough
2. Done one tool round-trip **by hand** (you are the loop)
3. Written the loop
4. Broken it on purpose and fixed it
5. Learned to read an agent trace
6. Compared your loop with real-world agents (Hermes, OpenClaw)

**No frameworks today.** Every line that talks to the model is code you can read.

## 0. Setup

We use **OpenAI** by default. The same notebook also runs on **Claude**: set `PROVIDER = "anthropic"` below.

| Provider | Key | Default model |
|---|---|---|
| OpenAI (default) | `OPENAI_API_KEY` | `gpt-6-luna` |
| Anthropic | `ANTHROPIC_API_KEY` (or `ant auth login`) | `claude-opus-5-5` |

**Your API key** goes in a file called `.env` in the course folder (once, for all weeks):
1. Copy `.env.example` to `.env`.
2. Replace `sk-proj-your-key-here` with your key and save.

`.env` is never committed to git. **Never paste a key into a notebook cell:** notebooks get shared.

**Alex's week is always the current week:** the setup cell writes `data/` from the template in the course's `life/`
folder, dated so that *today* in the simulation is this week's Monday. (To pin a date, put `LIFE_TODAY=2026-10-12`
in `.env`.)

First, make sure the packages are installed **in the Python this notebook runs on**:

In [ ]:
%pip install -q openai anthropic pydantic python-dotenv


In [3]:
import json, os
from pathlib import Path

from providers import load_api_key
from simweek import build_week

PROVIDER = "openai"            # "openai" or "anthropic"
MODELS = {"openai": "gpt-6-luna", "anthropic": "claude-opus-5-5"}
MODEL = MODELS[PROVIDER]
load_api_key(PROVIDER)             # reads your key from the course's .env file

DATA_DIR = Path("data").resolve()
WEEK = build_week(DATA_DIR)        # Alex's week, dated to the current week (see life/simweek.py)

if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI()                  # reads OPENAI_API_KEY
else:
    import anthropic
    client = anthropic.Anthropic()     # reads ANTHROPIC_API_KEY (or your `ant auth login` profile)

print(f"Using {PROVIDER} / {MODEL}")
print(f"Alex's week: {WEEK.label(0)} – {WEEK.label(6)}  (in the simulation, today is {WEEK.today})")
print("Data files:", sorted(p.name for p in DATA_DIR.iterdir()))



Using openai / gpt-6-luna
Alex's week: Monday, October 12 – Sunday, October 18  (in the simulation, today is Monday 2026-10-12)
Data files: ['calendar.csv', 'notes', 'tasks.csv']


## 1. One LLM call is not an agent

Ask the model something it **cannot know** (our private files) and something it **should not guess** (exact arithmetic).

In [5]:
QUESTION = (f"What's on my calendar on {WEEK.label(1)} (data/calendar.csv)? "
            "Also, what is 1234.56 * 987.65 + 0.07 ** 3 exactly?")

if PROVIDER == "openai":
    response = client.responses.create(model=MODEL, input=QUESTION)
    print("status:", response.status)
    print(response.output_text)
else:
    response = client.messages.create(model=MODEL, max_tokens=4000,
                                      messages=[{"role": "user", "content": QUESTION}])
    print("stop_reason:", response.stop_reason)
    print("".join(b.text for b in response.content if b.type == "text"))


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

**Discuss:** What did the model do with the file question? Check the arithmetic in Python (`1234.56 * 987.65 + 0.07 ** 3`). Was it exact?

The model has no eyes and no calculator. **Tools** give it both.

## 2. One tool, one round-trip — *you* are the loop

A tool has two halves:

| Half | Who uses it | What it is |
|---|---|---|
| **Schema** | the model | name + description + JSON schema of the arguments |
| **Function** | your code | the Python that actually runs |

The model never executes anything. It only **asks**.

We write tool schemas once, in a neutral shape (`name`, `description`, `input_schema`). Each API wraps it
slightly differently. Both versions are in the cells below, so you can compare them.

In [ ]:
calculator_schema = {
    "name": "calculator",
    "description": "Evaluate an arithmetic expression exactly. Supports + - * / ** % and parentheses.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(2 + 3) * 4'"}},
        "required": ["expression"],
    },
}
QUESTION = "What is 1234.56 * 987.65 + 0.07 ** 3? Use the calculator."

if PROVIDER == "openai":
    tools = [{"type": "function", "name": calculator_schema["name"],
              "description": calculator_schema["description"],
              "parameters": calculator_schema["input_schema"], "strict": False}]
    history = [{"role": "user", "content": QUESTION}]
    response = client.responses.create(model=MODEL, input=history, tools=tools)
    for item in response.output:                       # <- expect a "function_call" item
        print(item.type, "->", item.model_dump(exclude_none=True))
else:
    tools = [calculator_schema]
    history = [{"role": "user", "content": QUESTION}]
    response = client.messages.create(model=MODEL, max_tokens=4000, tools=tools, messages=history)
    print("stop_reason:", response.stop_reason)       # <- expect "tool_use"
    for block in response.content:
        print(block.type, "->", block.model_dump(exclude={"signature"}))


The model is saying: *"I'm not done. Please run this and tell me what happened."*
- **OpenAI:** a `function_call` item with `call_id`, `name` and `arguments`. Note that `arguments` is a **JSON string**.
- **Claude:** `stop_reason == "tool_use"` and a `tool_use` block with `id`, `name` and `input` (already a dict).

Let's write the function. **Never use `eval()` on model output**: that would let the model run any Python on your machine.
We walk the syntax tree and only allow arithmetic.

In [ ]:
import ast, operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression element: {ast.dump(node)}")

def calculator(expression: str) -> str:
    return str(round(_eval_node(ast.parse(expression, mode="eval").body), 6))

print(calculator("1234.56 * 987.65 + 0.07 ** 3"))


Now do by hand what the loop will do automatically:

1. Append the model's **whole** reply (all items, not just text) to the history
2. Run the tool
3. Send back a result that carries the **same id** as the request
4. Call the model again

In [ ]:
if PROVIDER == "openai":
    history += response.output                                                    # 1
    call = next(i for i in response.output if i.type == "function_call")
    output = calculator(**json.loads(call.arguments))                             # 2 (arguments is a string!)
    history.append({"type": "function_call_output", "call_id": call.call_id,      # 3
                    "output": output})
    response = client.responses.create(model=MODEL, input=history, tools=tools)   # 4
    print(response.output_text)
else:
    history.append({"role": "assistant", "content": response.content})           # 1
    call = next(b for b in response.content if b.type == "tool_use")
    output = calculator(**call.input)                                             # 2
    history.append({"role": "user", "content": [                                  # 3
        {"type": "tool_result", "tool_use_id": call.id, "content": output}]})
    response = client.messages.create(model=MODEL, max_tokens=4000, tools=tools, messages=history)  # 4
    print("stop_reason:", response.stop_reason)
    print("".join(b.text for b in response.content if b.type == "text"))


**Look at the shape of the history:**

In [ ]:
for entry in history:
    if isinstance(entry, dict):
        kind = entry.get("role") or entry.get("type")
        content = entry.get("content")
        parts = [c["type"] if isinstance(c, dict) else c.type for c in content] if isinstance(content, list) else []
        print(f"{kind:22} {parts}")
    else:
        print(f"{entry.type:22} (model output item)")


Same four steps, different envelopes:
- **OpenAI:** the history is a flat list of *items*: your message, the model's output items (reasoning, `function_call`),
  then your `function_call_output`.
- **Claude:** the history is a list of *messages*, and tool results travel in a **`user`** message. From the API's
  point of view, *the environment* speaks as the user.

**Question:** What if the model needed 7 tool calls instead of 1? → We need a loop.

## 3. A tool registry

To loop over *any* tool we need a lookup: `name → (schema, function)`.
Let's add two file tools. They are **sandboxed**: the model can only see inside `data/`.

In [ ]:
def _safe_path(path: str) -> Path:
    target = (DATA_DIR / path).resolve()
    if not target.is_relative_to(DATA_DIR):
        raise PermissionError(f"'{path}' is outside the data folder")
    return target

def list_dir(path: str = ".") -> str:
    target = _safe_path(path)
    if not target.is_dir():
        raise FileNotFoundError(f"No such directory: '{path}'")
    return "\n".join(f"{p.name}/" if p.is_dir() else p.name for p in sorted(target.iterdir())) or "(empty)"

def read_file(path: str) -> str:
    target = _safe_path(path)
    if not target.is_file():
        raise FileNotFoundError(f"No such file: '{path}'. Use list_dir to see what exists.")
    return target.read_text(encoding="utf-8")[:20_000]

TOOLS = {
    "list_dir": {"fn": list_dir, "schema": {
        "name": "list_dir",
        "description": "List files and subfolders in the data folder. Folders end with '/'. Call this first when you don't know which files exist.",
        "input_schema": {"type": "object", "properties": {
            "path": {"type": "string", "description": "Folder relative to the data folder. Use '.' for the root."}},
            "required": []}}},
    "read_file": {"fn": read_file, "schema": {
        "name": "read_file",
        "description": "Read a UTF-8 text file (e.g. CSV, TXT) from the data folder and return its full contents.",
        "input_schema": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File path relative to the data folder, e.g. 'calendar.csv'."}},
            "required": ["path"]}}},
    "calculator": {"fn": calculator, "schema": {
        **calculator_schema,
        "description": calculator_schema["description"] + " Put a whole sum in one expression, e.g. '3 + 6 + 2.5'."}},
}

print(list_dir())


`execute_tool` turns one tool call into one `ToolResult`.

**Key design decision:** errors are **returned to the model**, not raised. A crashed agent learns nothing; a model that reads `FileNotFoundError: ... Use list_dir` can fix its own mistake.

In [ ]:
from types import SimpleNamespace
from providers import ToolResult

def execute_tool(call, tools=TOOLS) -> ToolResult:
    tool = tools.get(call.name)
    if tool is None:
        return ToolResult(call.id, f"Unknown tool '{call.name}'. Available: {', '.join(tools)}", is_error=True)
    try:
        return ToolResult(call.id, str(tool["fn"](**call.input)))
    except Exception as exc:
        return ToolResult(call.id, f"{type(exc).__name__}: {exc}", is_error=True)

# Try it with a fake call — no API needed
print(execute_tool(SimpleNamespace(id="test", name="read_file", input={"path": "nope.csv"})))
print(execute_tool(SimpleNamespace(id="test", name="read_file", input={"path": "../../etc/passwd"})))


## 4. Close the loop

In Part 2 you saw that OpenAI and Claude wrap the same four steps in different envelopes. To write **one** loop,
`providers.py` (next to this notebook, read it!) hides the envelope behind three methods:

| Method | Does |
|---|---|
| `conv.ask()` | calls the model with the history so far, appends the whole reply, returns a `Step` |
| `conv.add_tool_results(results)` | appends our results in the provider's format |
| `step.stop`, `step.text`, `step.tool_calls` | what the model said, translated into one language |

The **loop itself** is ours. Every `step.stop` value is a branch:

| `step.stop` | Meaning | What the loop does |
|---|---|---|
| `tool_use` | "run these tools for me" | execute all, append results, **continue** |
| `end_turn` | "I'm done" | return the answer |
| `max_tokens` | reply was cut off | stop: don't run tool calls with possibly truncated arguments |
| `refusal` | the model declined | stop and report |

Plus one condition **we** own: `max_turns`. The model decides *when* it is done; we decide *how long we're willing to wait*.

In [ ]:
import inspect, providers
print(inspect.getsource(providers.ResponsesConversation.ask))      # how the OpenAI envelope is translated


In [ ]:
from dataclasses import dataclass, field
from providers import open_conversation

SYSTEM_PROMPT = (
    "You are a personal assistant for Alex, a 44-year-old software developer with a full-time job and "
    f"two kids, who is retraining to become an AI engineer. Today is {WEEK.today}. "
    "Alex's calendar, task list and notes are files in a sandboxed data folder. Never do arithmetic "
    "in your head: use the calculator. When you have the answer, reply briefly and show the key "
    "numbers you used."
)

@dataclass
class Result:
    answer: str
    stop: str
    turns: int
    usage: list = field(default_factory=list)      # (input_tokens, output_tokens) per turn

def _short(x, n=200):
    s = str(x).replace("\n", " | ")
    return s if len(s) <= n else s[:n] + "..."

def trace(turn, step, label=""):
    print(f"\n── turn {turn}{label} · stop={step.stop} · in={step.input_tokens} out={step.output_tokens} tokens")
    if step.text:
        print("  💬", _short(step.text, 300))
    for c in step.tool_calls:
        print(f"  🔧 {c.name}({c.input})")

def run_agent(task, tools=TOOLS, max_turns=10, provider=None, model=None, effort=None, verbose=True):
    provider, model = provider or PROVIDER, model or MODEL
    conv = open_conversation(provider, model, task, tools=[t["schema"] for t in tools.values()],
                             system=SYSTEM_PROMPT, effort=effort)
    usage = []

    for turn in range(1, max_turns + 1):
        step = conv.ask()                         # the whole reply is appended inside: history is append-only
        usage.append((step.input_tokens, step.output_tokens))
        if verbose:
            trace(turn, step)

        if step.stop == "tool_use":
            results = [execute_tool(c, tools) for c in step.tool_calls]
            if verbose:
                for r in results:
                    print("  ❌" if r.is_error else "  ✅", _short(r.content))
            conv.add_tool_results(results)        # ALL results go back together
            continue

        answer = step.text
        if step.stop == "max_tokens":
            answer += "\n[cut off: max_tokens]"
        elif step.stop == "refusal":
            answer = "[the model declined this request]"
        return Result(answer, step.stop, turn, usage)

    return Result(f"[stopped after {max_turns} turns without a final answer]", "max_turns", max_turns, usage)


In [ ]:
TASK = ("Which open tasks are due before my capstone demo on Friday, how many hours of work are they in "
        "total, and how much free time do I have between 19:00 and 22:30 from Monday to Thursday? "
        "Is it enough?")

result = run_agent(TASK)
print("\n" + "═" * 60 + "\n" + result.answer)


**Never trust, verify.** Here is the ground truth computed with plain Python. Does the agent agree?

In [ ]:
import csv

def hours(hhmm):
    h, m = map(int, hhmm.split(":"))
    return h + m / 60

events = list(csv.DictReader(open(DATA_DIR / "calendar.csv")))
tasks = list(csv.DictReader(open(DATA_DIR / "tasks.csv")))

due = [t for t in tasks if t["status"] == "todo" and t["due"] < WEEK.day(4)]          # before Friday
need = sum(float(t["est_hours"]) for t in due)
print("Due before Friday:", ", ".join(t["title"] for t in due), f"→ {need} h of work")

total_free = 0
for day in [WEEK.day(i) for i in range(4)]:                                            # Monday-Thursday
    # clip events to the 19:00-22:30 evening window, then MERGE overlapping ones
    busy = sorted((max(hours(e["start"]), 19), min(hours(e["end"]), 22.5)) for e in events if e["date"] == day)
    merged = []
    for start, end in (b for b in busy if b[1] > b[0]):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    free = 3.5 - sum(end - start for start, end in merged)
    total_free += free
    print(f"{day}: {free} h free")
print(f"Free time Mon-Thu: {total_free} h  vs  {need} h needed  →  margin {total_free - need} h")


**Discuss:**
- How many turns did it take? Did it call several tools in **one** turn (parallel tool use)?
- Tuesday evening has two **overlapping** events (AI course session 19:30–21:00, recruiter screen 20:30–21:00).
  If the agent just added up event durations, its free time is off by 0.5 h. Did it notice? (Did *Alex*?)
- Did it read `notes/preferences.txt` or the syllabus? Should it have?
- "Is it enough?" **No:** 9.25 h of work against 8.25 h of free evenings. What should the assistant say next?
  Use the early mornings (see `notes/preferences.txt`)? Drop the LinkedIn update? Ask for help at home?
  Who decides? (Ambiguity in the *task* is not a loop bug.)
- Switch `PROVIDER` and run it again. Same answer? Same number of turns?

## 5. Break it on purpose

Agents fail in ways normal programs don't. Run each experiment, then write down **what you expected** vs **what happened**.

### 5a. A tool fails → does the agent recover?

In [ ]:
print(run_agent("What's on my calendar in calendar_november.csv?").answer)


Look for the ❌ line. Did the model call `list_dir` next? Did it make up November events, or say the file doesn't exist?

**Try:** change `execute_tool` to `raise` instead of returning the error. What happens to the whole run?

### 5b. The model wants more turns than we allow

In [ ]:
result = run_agent(TASK, max_turns=2)
print("\n" + result.answer)
print("stop:", result.stop, "| turns:", result.turns)


**Discuss:** Is returning `"[stopped after 2 turns]"` good enough? Alternatives:
- ask the model for a best-effort summary with tools disabled (one more call with no `tools`)
- warn the model *before* the budget runs out (Hermes does this, see Part 7)
- hand control back to a human

### 5c. Tool descriptions are prompts

The model chooses tools **only** from name + description + schema. Make them bad and watch.

In [ ]:
import copy

BAD_TOOLS = copy.deepcopy(TOOLS)
BAD_TOOLS["read_file"]["schema"]["description"] = "Gets things."
BAD_TOOLS["read_file"]["schema"]["input_schema"]["properties"]["path"]["description"] = "the thing"
BAD_TOOLS["list_dir"]["schema"]["description"] = "Misc."
# Rename read_file -> fetch (the dict key stays the lookup name, so rename both)
BAD_TOOLS["fetch"] = BAD_TOOLS.pop("read_file")
BAD_TOOLS["fetch"]["schema"]["name"] = "fetch"

print(run_agent("How many hours of open tasks do I have?", tools=BAD_TOOLS).answer)


Compare the trace with the good tools. Count turns, look for wrong arguments (URLs? full paths?).

**And without the calculator?**

In [ ]:
NO_CALC = {k: v for k, v in TOOLS.items() if k != "calculator"}
print(run_agent("How many hours of open tasks do I have in total? Give the exact number.", tools=NO_CALC).answer)


The exact answer is **11.75 h** (8 open tasks). Adding up decimals in its head: exact or not? Run it a few times.

## 6. Read the trace: where do the tokens go?

The API is **stateless**: every turn re-sends the *whole* history. Look at input tokens per turn.

In [ ]:
result = run_agent(TASK, verbose=False)
print("turn | input tokens | output tokens")
for turn, (tin, tout) in enumerate(result.usage, 1):
    print(f"{turn:4} | {tin:12,} | {tout:13,}")
print(f"total input: {sum(i for i, _ in result.usage):,}   total output: {sum(o for _, o in result.usage):,}")


**Discuss:** Input tokens grow every turn. If each turn adds *k* tokens, the total cost after *n* turns grows like *n²*.
That is why real agents invest so much in **context management** (compaction, caching, clearing old tool results), a later week.

## 7. Loops in the wild

Our loop is ~35 lines. Real agents keep exactly the same core and wrap a lot of engineering around it.
Read **`LOOPS_IN_THE_WILD.md`** (same folder). It walks through the real loop code of **Hermes Agent** (Nous Research, Python) and **OpenClaw** (TypeScript, adapted from the pi agent).

| | Our loop | Hermes Agent | OpenClaw / pi |
|---|---|---|---|
| Core | `while turn < max_turns` | `while api_calls < max_iterations and budget.remaining > 0` | `while (hasMoreToolCalls \|\| pendingMessages.length > 0)` inside `while (true)` |
| Who stops it | model (`end_turn`) or turn cap | model, iteration budget, wall-clock budget, interrupt | model, abort signal, hooks (`shouldStopAfterTurn`), loop detector |
| Tool errors | returned to the model | returned to model; invalid tool names answered with an error result | returned to model; truncated calls are failed, not run |
| Human mid-run | ✗ | interrupt + redirect | *steering* & *follow-up* message queues |
| Durability | none (in memory) | persists the tool call **before** executing it | event stream (`turn_start`, `message_end`, ...) |
| Providers | adapters in `providers.py` | many, via one internal (OpenAI-style) message format | many, via a provider layer |
| Lines | ~35 | ~1,800 in the loop module + per-phase modules | ~940 (pi) / ~1,450 (OpenClaw) |

**Discussion questions**
1. Strip Hermes and OpenClaw down to the lines that *must* be there. What's left? Is it our loop?
2. Each extra feature exists because something went wrong in production. Pick one and guess the incident.
3. Hermes appends a "budget almost used" warning to the newest tool result instead of the system prompt. Why there?
4. OpenClaw marks a turn as *tainted* once a tool returned content from the network. What attack is that preparing for?
5. Which of their features would you add to **our** loop first, and why?

## 8. Homework

### 8.1 From notebook to modules (required)

A notebook is a lab bench, not a product. Move your code into a package with this layout:

```
my_agent/
├── agent/
│   ├── __init__.py      # exports run_agent
│   ├── config.py        # PROVIDER, MODEL, MAX_TURNS, DATA_DIR, SYSTEM_PROMPT
│   ├── providers.py     # copy it from this folder: the adapters
│   ├── tools.py         # tool functions, schemas, TOOLS registry, execute_tool
│   ├── tracing.py       # trace, trace_tool_result
│   ├── loop.py          # run_agent: talks to the model only through providers.py
│   └── router.py        # (if you did Part 9) classify, choose_model, run_routed
├── tests/
│   └── test_agent.py    # tests that run WITHOUT an API key
└── main.py              # python main.py "your task"
```

Rules:
- `loop.py` must not know which tools exist; it receives `tools` as a parameter.
- `run_agent` accepts a `client=` (or `conversation=`) parameter so tests can pass a **fake** that returns scripted responses.
- At least these tests: calculator rejects `__import__('os')`; `read_file('../x')` is blocked;
  the loop stops at `max_turns`; two tool calls in one reply → both results go back before the next call.

### 8.2 A tool with side effects (required)

Add `write_file(path, content)`. Before it runs, print the path and the first lines of content and ask
`Allow? [y/N]` with `input()`. If the user says no, return an error result saying *the user declined*. Don't crash.
Then ask the agent to *"write a plan for my week to week_plan.md"*.

### 8.3 Stretch
- Stop the run when total tokens exceed a budget you choose; report the cost in dollars.
- When `max_turns` is hit, make one last call **without tools** asking for a best-effort answer.
- Add a budget warning like Hermes: at 80% of `max_turns`, tell the model to wrap up.
- After Part 9: add **escalation**. Start on the routed model; if two tool errors happen in a row or 70% of
  `max_turns` is used, continue the *same task* on the next stronger model.

## 9. (Optional) Pick the model before the loop: routing across providers

Not every task needs the strongest model. Listing a folder doesn't need the same model as a multi-file analysis.
Idea: **classify the task with a cheap model first**, then run the loop with the model that fits.

```
task ──► cheap classifier ──► tier: simple | moderate | hard
                                   │
                ROUTES[tier] = [candidates in order of preference]
                                   │  (first one that is available)
                                   ▼
               provider + model ──► the same agentic loop
```

Two different kinds of control flow sit side by side here:
- **The router is a workflow.** Our code decides the steps: classify → look up → run.
- **The loop is an agent.** The model decides the steps.

We can route across **three providers**: OpenAI, Claude (Anthropic), and a local model via **Ollama**.
Use whatever you have; providers you don't have are skipped automatically.

| Provider | What you need |
|---|---|
| OpenAI | `OPENAI_API_KEY` |
| Anthropic | `ANTHROPIC_API_KEY` (or `ant auth login`) |
| Ollama | Ollama ≥ 0.13.3 running locally, plus a tool-capable model: `ollama pull qwen3:8b` |

### 9.1 Why `providers.py` makes this easy

You've used adapters since Part 4. Here is the full comparison of the two wire formats they translate:

|  | OpenAI Responses API (OpenAI **and** Ollama) | Anthropic Messages API (Claude) |
|---|---|---|
| Tool definition | `{type: "function", name, description, parameters}` | `{name, description, input_schema}` |
| Model asks | output item `function_call`: `call_id`, `name`, `arguments` (**JSON string**) | content block `tool_use`: `id`, `name`, `input` (dict) |
| We answer | input item `function_call_output`: `call_id`, `output` (**no error flag**) | `user` message with `tool_result` blocks: `tool_use_id`, `content`, `is_error` |
| "Run my tools" | any `function_call` in `output` | `stop_reason == "tool_use"` |
| Cut off | `status == "incomplete"` | `stop_reason == "max_tokens"` |
| History | append `response.output` (incl. reasoning items) | append `response.content` |

Because the loop only sees `Step`, switching provider is one argument:

In [ ]:
from providers import is_available

for provider, model in [("openai", "gpt-6-luna"), ("anthropic", "claude-sonnet-5-5")]:
    if is_available(provider, model):
        r = run_agent("How many hours of open tasks do I have?", provider=provider, model=model, verbose=False)
        print(f"{provider}/{model}: {r.turns} turns → {_short(r.answer, 150)}")
    else:
        print(f"{provider}/{model}: no key, skipped")


### 9.2 The classifier

A cheap, fast model reads the task and returns a **tier**. Two rules:

1. **Constrain the output.** We use structured outputs (a JSON schema with an `enum`), so the answer is always parseable.
2. **Never trust it blindly.** If the call fails or returns something unexpected, fall back to the **strongest** tier.
   A hard task on a weak model is a worse failure than an easy task on a strong one.

In [ ]:
TIERS = ("simple", "moderate", "hard")
DEFAULT_TIER = "hard"
CLASSIFIER = ("openai", "gpt-6-luna") if PROVIDER == "openai" else ("anthropic", "claude-haiku-4-5")
# try ("ollama", "qwen3:8b") for a free, private classifier

CLASSIFIER_PROMPT = '''You route tasks for a personal assistant. The assistant has tools to list and read files
(calendar, tasks, notes) and a calculator. Classify how demanding the task is FOR THAT AGENT:

- simple:   one or two tool calls, no real reasoning (list files, read one file, one sum)
- moderate: several tool calls and straightforward arithmetic over one or two files
- hard:     multi-step reasoning across several files, time arithmetic, judgement or ambiguity

Answer with the tier and a one-sentence reason.'''

CLASSIFIER_SCHEMA = {
    "type": "object",
    "properties": {"tier": {"type": "string", "enum": list(TIERS)}, "reason": {"type": "string"}},
    "required": ["tier", "reason"],
    "additionalProperties": False,
}

def classify(task):
    provider, model = CLASSIFIER
    try:
        answer = open_conversation(provider, model, task, system=CLASSIFIER_PROMPT).complete_json(CLASSIFIER_SCHEMA)
        if answer.get("tier") in TIERS:
            return answer["tier"], answer.get("reason", "")
        return DEFAULT_TIER, f"unknown tier {answer.get('tier')!r}"
    except Exception as exc:
        return DEFAULT_TIER, f"classifier failed ({type(exc).__name__}: {exc})"

TASKS = [
    "List the files in the data folder.",
    "How many hours of open tasks do I have?",
    TASK,   # the multi-file free-time question from Part 4
]
for t in TASKS:
    print(classify(t), "←", t[:60])


### 9.3 The route table

For each tier, a list of candidates **in order of preference**. The router takes the first one that's available.
If a whole tier is unavailable it tries **stronger** tiers first, and weaker ones only as a last resort.

> Model names change often. Check each provider's model list and edit the table. It's data, not code.

In [ ]:
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen3:8b")

ROUTES = {
    "simple":   [("ollama", OLLAMA_MODEL), ("openai", "gpt-6-luna"), ("anthropic", "claude-haiku-4-5")],
    "moderate": [("openai", "gpt-6.1-sol"), ("anthropic", "claude-sonnet-5-5")],
    "hard":     [("openai", "gpt-6-astra"), ("anthropic", "claude-opus-5-5")],
}

def choose_model(tier):
    i = TIERS.index(tier)
    for t in TIERS[i:] + TIERS[:i][::-1]:            # this tier, stronger tiers, then weaker
        for provider, model in ROUTES[t]:
            if is_available(provider, model):
                return provider, model
    raise RuntimeError("No configured model is available")

for tier, candidates in ROUTES.items():
    print(f"{tier:9}", "  ".join(f"{'✓' if is_available(p, m) else '✗'} {p}/{m}" for p, m in candidates))


### 9.4 Route, then loop

In [ ]:
def run_routed(task, verbose=True):
    tier, reason = classify(task)
    provider, model = choose_model(tier)
    if verbose:
        print(f"\n🧭 tier={tier} → {provider}/{model}   ({reason})")
    r = run_agent(task, provider=provider, model=model, verbose=False)
    return {"tier": tier, "model": f"{provider}/{model}", "turns": r.turns,
            "in": sum(i for i, _ in r.usage), "out": sum(o for _, o in r.usage), "answer": r.answer}

routed = [run_routed(t) for t in TASKS]
for r in routed:
    print(f"\n{r['tier']:9} {r['model']:32} {r['turns']} turns  {r['in']:>6} in {r['out']:>5} out\n   {_short(r['answer'], 160)}")


### 9.5 Experiment: is routing worth it?

Routing only pays off if it saves more than it costs. Run the same tasks three ways and fill in the table:

| Setup | What it is |
|---|---|
| **A. Always strongest** | the "hard" model, default effort, for every task |
| **B. Routed** | classifier + route table (above) |
| **C. One model, effort tuned** | the "hard" model with effort `low` for every task (no classifier, one model) |

Prices are per million tokens. Fill in the ones marked `None` from the provider's pricing page. Local models cost 0 (but not *nothing*: your hardware and time).

In [ ]:
PRICES = {   # (input $, output $) per 1M tokens
    "openai/gpt-6-luna": (0.10, 0.50),
    "openai/gpt-6.1-sol": None,
    "openai/gpt-6-astra": None,
    "anthropic/claude-haiku-4-5": (1.00, 5.00),
    "anthropic/claude-sonnet-5-5": (2.00, 10.00),
    "anthropic/claude-opus-5-5": (4.00, 20.00),
    f"ollama/{OLLAMA_MODEL}": (0.0, 0.0),
}

def cost(model, tin, tout):
    p = PRICES.get(model)
    return None if p is None else (tin * p[0] + tout * p[1]) / 1e6

STRONG = choose_model("hard")

def run_fixed(task, effort=None):
    r = run_agent(task, provider=STRONG[0], model=STRONG[1], effort=effort, verbose=False)
    return {"model": "/".join(STRONG), "turns": r.turns, "in": sum(i for i, _ in r.usage),
            "out": sum(o for _, o in r.usage), "answer": r.answer}

rows = []
for i, t in enumerate(TASKS):
    a, c = run_fixed(t), run_fixed(t, effort="low")
    b = routed[i]
    for name, r in (("A strongest", a), ("B routed", b), ("C strong+low", c)):
        rows.append((i + 1, name, r["model"], r["turns"], cost(r["model"], r["in"], r["out"]), r["answer"]))

print("task | setup        | model                          | turns | cost $")
for task_no, name, model, turns, usd, _ in rows:
    print(f"{task_no:4} | {name:12} | {model:30} | {turns:5} | {'?' if usd is None else f'{usd:.4f}'}")


Now check **correctness** against the ground truth from Part 4. Cheap but wrong is not cheap.

```python
for task_no, name, model, *_, answer in rows: print(task_no, name, "→", _short(answer, 200))
```

**Discuss:**
1. Add the **classifier's** cost and latency to setup B. On which tasks does routing still win?
2. Did any task get misrouted? What did it cost: a wrong answer, or more turns?
3. Setup C has no classifier and one model. How close does it get to B? (Prompt caches only work within one model,
   so switching models loses them. One model + effort tuning is often the simpler win.)
4. The local model: did it call the tools correctly? Small models often struggle with tool calling, so the "cheap" tier can need more turns.
5. **Routing vs escalation.** We choose once, *before* the loop. OpenClaw's `prepareNextTurn` hook can switch models
   *between turns* (see `LOOPS_IN_THE_WILD.md`). When would "start cheap, escalate if stuck" beat classifying up front?
6. Hermes supports many providers by converting everything into one internal message format.
   `providers.py` keeps each provider's native format inside its adapter instead. What are the trade-offs?